# Etapa 1: Extração e Tratamento de Dados - RAIS 2023 (Espírito Santo)

Este notebook tem como objetivo processar os microdados brutos da RAIS 2023, selecionando apenas as variáveis de interesse para o case da Porto Verde Capital. Isso garantirá um arquivo leve e otimizado para a construção do dashboard no Power BI.

In [7]:
import pandas as pd
import numpy as np

### 1.1 Mapeamento das Colunas-Chave

Com base no dicionário de dados (`RAIS_vinculos_layout2020.xls`) e nas necessidades da gestora Maria Fernandes, selecionamos as seguintes colunas:
* **Geografia:** `municpio` (município do estabelecimento)
* **Setor/Atividade:** `ibgesubsetor`, `cnae20classe`
* **Ocupação:** `cboocupao2002`
* **Demografia:** `idade`, `faixaetria`, `sexotrabalhador`
* **Remuneração:** `vlremunmdianom`
* **Filtro:** `vnculoativo3112` (para manter apenas quem estava empregado no fim do ano)

In [8]:
colunas_interesse = [
    'municpio', 'ibgesubsetor', 'cnae20classe', 'cboocupao2002', 
    'sexotrabalhador', 'idade', 'faixaetria', 'vlremunmdianom', 
    'vnculoativo3112'
]

### 1.2 Leitura dos Dados

Como o arquivo original é muito pesado, vamos carregar apenas as colunas mapeadas acima utilizando o parâmetro `usecols`. Lemos inicialmente como texto (`dtype=str`) para evitar a perda de formatação original (como os zeros à esquerda nos códigos CNAE e CBO).

In [9]:
caminho_arquivo = 'ES2023.txt'

print("Iniciando a leitura do arquivo...")
df_rais = pd.read_csv(
    caminho_arquivo, 
    sep=';', 
    encoding='latin1', 
    usecols=colunas_interesse, 
    dtype=str
)

print(f"Leitura concluída! O dataset possui {df_rais.shape[0]} linhas e {df_rais.shape[1]} colunas.")
display(df_rais.head())

Iniciando a leitura do arquivo...
Leitura concluída! O dataset possui 1700397 linhas e 9 colunas.


,cboocupao2002,cnae20classe,vnculoativo3112,faixaetria,idade,municpio,vlremunmdianom,sexotrabalhador,ibgesubsetor
0,621005,1342,0,6,40,320320,"00000000000001307,05",1,25
1,621005,1342,0,6,42,320320,"00000000000001314,56",1,25
2,621005,1512,0,7,51,320150,"00000000000001464,92",1,25
3,622605,1351,0,6,47,320320,"00000000000000000,00",1,25
4,622020,1342,0,7,56,320501,"00000000000001096,92",1,25


### 1.3 Limpeza e Transformação

Nesta etapa, vamos:
1. Filtrar apenas os trabalhadores com vínculo ativo em 31/12 para uma fotografia real do emprego no fim do ano.
2. Converter a coluna de remuneração nominal para formato numérico.
3. Converter as variáveis de idade e sexo para números.

In [10]:
# 1. Filtrar apenas vínculos ativos e resetar o índice para começar do 0
df_ativos = df_rais[df_rais['vnculoativo3112'] == '1'].copy().reset_index(drop=True)
print(f"Trabalhadores ativos em 31/12 no ES: {df_ativos.shape[0]}")

# Descartar a coluna de filtro, pois todos agora são '1'
df_ativos.drop(columns=['vnculoativo3112'], inplace=True)

# 2. Tratamento da Remuneração (substituir vírgula por ponto e converter para float)
df_ativos['vlremunmdianom'] = df_ativos['vlremunmdianom'].str.replace(',', '.', regex=False)
df_ativos['vlremunmdianom'] = pd.to_numeric(df_ativos['vlremunmdianom'], errors='coerce')

# 3. Conversão de Idade e Sexo para numérico
df_ativos['idade'] = pd.to_numeric(df_ativos['idade'], errors='coerce')
df_ativos['sexotrabalhador'] = pd.to_numeric(df_ativos['sexotrabalhador'], errors='coerce')

display(df_ativos.head())

Trabalhadores ativos em 31/12 no ES: 1085430


,cboocupao2002,cnae20classe,faixaetria,idade,municpio,vlremunmdianom,sexotrabalhador,ibgesubsetor
0,514225,84116,7,59,320517,3113.0,1,24
1,621005,1512,7,51,320150,1321.0,1,25
2,621005,1512,7,52,320150,1753.0,1,25
3,514320,10121,6,44,320140,1268.0,1,13
4,783225,47440,7,59,320480,1459.0,1,16


### 1.4 Tradução dos Códigos (Extração Dinâmica do Dicionário)

Para que o painel no Power BI seja analítico e não apenas uma tabela de códigos numéricos, precisamos enriquecer a base.
Para isso, vamos ler o arquivo `RAIS_vinculos_layout2020.xls` e extrair de-paras de forma totalmente dinâmica, evitando *hardcoding*.

In [11]:
# Lendo o dicionário (Certifique-se de que o arquivo xls está na mesma pasta)
dicionario_xls = pd.ExcelFile('RAIS_vinculos_layout2020.xls')

# 1. Lendo a aba de layout geral (onde estão Sexo e Subsetor IBGE)
# O cabeçalho real das variáveis começa na linha 4 (index 3 do pandas)
df_layout = pd.read_excel(dicionario_xls, sheet_name='RAISD - layout', header=3)

# Como as células no Excel são mescladas, preenchemos os nomes das variáveis para baixo
df_layout['Nome'] = df_layout['Nome'].ffill()

# --- A. Traduzindo Sexo ---
df_sexo_layout = df_layout[df_layout['Nome'] == 'SEXO'].dropna(subset=['Categorias', 'Valor na Fonte'])
# Criando o dicionário {1: 'MASCULINO', 2: 'FEMININO'}
dict_sexo = dict(zip(pd.to_numeric(df_sexo_layout['Valor na Fonte'], errors='coerce'), df_sexo_layout['Categorias']))
df_ativos['nome_sexo'] = df_ativos['sexotrabalhador'].map(dict_sexo)

# --- B. Traduzindo Subsetor IBGE ---
df_subsetor_layout = df_layout[df_layout['Nome'] == 'IBGE Subsetor'].dropna(subset=['Categorias', 'Valor na Fonte'])
dict_subsetor = dict(zip(pd.to_numeric(df_subsetor_layout['Valor na Fonte'], errors='coerce'), df_subsetor_layout['Categorias']))
df_ativos['nome_setor_macro'] = pd.to_numeric(df_ativos['ibgesubsetor'], errors='coerce').map(dict_subsetor)

print("Gênero e Subsetor Macroeconômico mapeados dinamicamente com sucesso!")

Gênero e Subsetor Macroeconômico mapeados dinamicamente com sucesso!


### 1.5 Extraindo Atividades, Ocupações e Municípios (Abas Específicas)

In [ ]:
# --- F. Traduzindo Faixa Etária ---
df_faixas = pd.read_excel(dicionario_xls, sheet_name='FAIXAS')
df_faixa_etaria = df_faixas.iloc[0:8, 0:2].copy() 
df_faixa_etaria.columns = ['faixaetria', 'nome_faixa_etaria']

# CORREÇÃO: Forçando ambos os lados a serem tratados como números para o cruzamento bater perfeitamente
df_faixa_etaria['faixaetria'] = pd.to_numeric(df_faixa_etaria['faixaetria'], errors='coerce')
df_ativos['faixaetria'] = pd.to_numeric(df_ativos['faixaetria'], errors='coerce')

df_ativos = df_ativos.merge(df_faixa_etaria, on='faixaetria', how='left')

# Visualizando como ficou a relação final com TUDO incluído
colunas_exibicao = [
    'nome_municipio', 'nome_setor_macro', 'nome_atividade_cnae', 
    'nome_ocupacao', 'nome_sexo', 'idade', 'nome_faixa_etaria', 'vlremunmdianom'
]
display(df_ativos[colunas_exibicao].head())

,nome_municipio,nome_setor_macro,nome_atividade_cnae,nome_ocupacao,nome_sexo,idade,nome_faixa_etaria,vlremunmdianom
0,Vila Valerio,Administraçao pública direta e autárquica,Administração Pública em Geral,Trabalhador de Servicos de Limpeza e Conservac...,MASCULINO,59,NaN,3113.0
1,Colatina,"Agricultura, silvicultura, criaçao de animais,...",NaN,Trabalhador Agropecuario em Geral,MASCULINO,51,NaN,1321.0
2,Colatina,"Agricultura, silvicultura, criaçao de animais,...",NaN,Trabalhador Agropecuario em Geral,MASCULINO,52,NaN,1753.0
3,Castelo,"Indústria de produtos alimentícios, bebidas e ...",Abate de Aves,Faxineiro,MASCULINO,44,NaN,1268.0
4,Sao Jose do Calcado,Comércio varejista,Comércio Varejista de Ferragens e Ferramentas,Ajudante de Motorista,MASCULINO,59,NaN,1459.0
